In [0]:
import json
import time
import datetime as dt
from typing import Optional, Dict, Any

import requests

In [0]:
# ---------------- Config ----------------
CATALOG = "canada_business"   # <-- change if needed
SCHEMA  = "bronze"            # <-- your bronze UC schema

BASE_URL = "https://api.stlouisfed.org/fred/series/observations"

SERIES: Dict[str, str] = {
    "DCOILWTICO": "Crude Oil Prices: WTI (Daily)",
    "DGS10": "10-Year Treasury Constant Maturity Rate (Daily)",
    "FEDFUNDS": "Effective Federal Funds Rate (Daily)",
    "T5YIE": "5-Year Breakeven Inflation Rate (Daily)",
    "ICSA": "Initial Claims (Weekly)",
    "IC4WSA": "4-Week Moving Avg Initial Claims (Weekly)",
    "DEXUSEU": "US/Euro FX Rate (Daily)",
    "DEXCAUS": "Canada/US FX Rate (Daily)",
}

In [0]:
# Secret stored in Databricks secret scope backed by Azure Key Vault
SECRET_SCOPE = "fred-api-scope"
SECRET_KEY   = "fredkey"
FRED_API_KEY = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)

# Bronze subfolder under your UC-managed schema location
BRONZE_SUBDIR = "fred_raw"

# Incremental mode (recommended):
# - None means "full history" (bigger files; not truly incremental)
# - Or set DAYS_BACK = 30 for last 30 days each run (simple incremental-ish demo)
DAYS_BACK: Optional[int] = 30   # change to None if you want full history


In [0]:
# ---------------- Helpers ----------------
def run_ts_utc() -> str:
    return dt.datetime.now(dt.timezone.utc).strftime("%Y%m%dT%H%M%SZ")


def get_observation_start(days_back: Optional[int]) -> Optional[str]:
    if days_back is None:
        return None
    start_date = (dt.datetime.now(dt.timezone.utc) - dt.timedelta(days=days_back)).date()
    return start_date.isoformat()


def get_json_with_retry(
    url: str,
    params: Dict[str, Any],
    headers: Optional[Dict[str, str]] = None,
    retries: int = 4,
    timeout: int = 30
) -> Dict[str, Any]:
    backoff = 1.0
    last_err: Optional[Exception] = None

    for attempt in range(1, retries + 1):
        try:
            r = requests.get(url, params=params, headers=headers, timeout=timeout)

            # Retry on transient or rate limit
            if r.status_code in (429, 500, 502, 503, 504):
                raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")

            r.raise_for_status()
            return r.json()

        except Exception as e:
            last_err = e
            if attempt == retries:
                break
            time.sleep(backoff)
            backoff *= 2

    raise RuntimeError(f"Request failed after {retries} attempts: {last_err}")


def fetch_fred_series(series_id: str, observation_start: Optional[str]) -> Dict[str, Any]:
    params: Dict[str, Any] = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
    }
    if observation_start:
        params["observation_start"] = observation_start

    headers = {
        "User-Agent": "databricks-lakehouse-demo/1.0"
    }

    return get_json_with_retry(BASE_URL, params=params, headers=headers)


def resolve_schema_root_location(catalog: str, schema: str) -> str:
    df = spark.sql(f"DESCRIBE SCHEMA EXTENDED {catalog}.{schema}").toPandas()

    rows = df.loc[
        df["database_description_item"] == "RootLocation",
        "database_description_value"
    ].values

    if len(rows) == 0 or not rows[0]:
        raise RuntimeError(
            f"RootLocation not found for {catalog}.{schema}. "
            f"Ensure schema was created with MANAGED LOCATION."
        )

    return str(rows[0])


def join_uri(base_uri: str, child: str) -> str:
    # safe join for abfss:// URIs
    return base_uri.rstrip("/") + "/" + child.strip("/")



In [0]:
# ---------------- Resolve UC schema RootLocation ----------------
root_location = resolve_schema_root_location(CATALOG, SCHEMA)
bronze_base = join_uri(root_location, BRONZE_SUBDIR)

data_base = bronze_base.rstrip("/") + "/data"
manifest_base = bronze_base.rstrip("/") + "/manifests"


ts = run_ts_utc()
observation_start = get_observation_start(DAYS_BACK)

print("Catalog.Schema:", f"{CATALOG}.{SCHEMA}")
print("RootLocation:", root_location)
print("Bronze base:", bronze_base)
print("Run ts:", ts)
print("Observation start:", observation_start if observation_start else "(full history)")

# ---------------- Ingest (write raw JSON files) ----------------
manifest = {
    "run_ts": ts,
    "catalog": CATALOG,
    "schema": SCHEMA,
    "bronze_base": bronze_base,
    "observation_start": observation_start,
    "series": []
}

# Ensure base dir exists
dbutils.fs.mkdirs(data_base)
dbutils.fs.mkdirs(manifest_base)


for series_id, desc in SERIES.items():
    payload = fetch_fred_series(series_id, observation_start=observation_start)

    out_dir  = join_uri(data_base, series_id)

    out_file = join_uri(out_dir, f"{series_id}_observations_{ts}.json")

    dbutils.fs.mkdirs(out_dir)

    # Use overwrite=False to prevent accidental clobbering
    # (file name already includes ts, so it should be unique)
    dbutils.fs.put(out_file, json.dumps(payload, indent=2), overwrite=False)

    obs_count = len(payload.get("observations", []))
    print(f"✅ {series_id} -> {obs_count} obs -> {out_file}")

    manifest["series"].append({
        "series_id": series_id,
        "description": desc,
        "rows": obs_count,
        "raw_file": out_file
    })

manifest_path = join_uri(manifest_base, f"run_manifest_{ts}.json")

dbutils.fs.put(manifest_path, json.dumps(manifest, indent=2), overwrite=False)
print("✅ Manifest:", manifest_path)